In [ ]:
# pre-processing
## from jupyter notebook - select individuals studied
for c in 1 2 3 4 5
do
  echo "=== chr${c}: norm + set-id + plink ==="

  bcftools norm -m -any -Oz \
    -o gt_only.present.chr${c}.norm.vcf.gz \
    gt_only.present.chr${c}.vcf.gz
  bcftools index -t gt_only.present.chr${c}.norm.vcf.gz

  bcftools annotate -Oz \
    -o gt_only.present.chr${c}.norm.id.vcf.gz \
    --set-id '%CHROM:%POS:%REF:%ALT' \
    gt_only.present.chr${c}.norm.vcf.gz
  bcftools index -t gt_only.present.chr${c}.norm.id.vcf.gz

  plink --vcf gt_only.present.chr${c}.norm.id.vcf.gz \
        --double-id \
        --make-bed \
        --out gt_only.present.chr${c}
done


printf "gt_only.present.chr2\ngt_only.present.chr3\ngt_only.present.chr4\ngt_only.present.chr5\n" > merge_list.chr2_5.txt

## Merge

plink --bfile gt_only.present.chr1 \
      --merge-list merge_list.chr2_5.txt \
      --make-bed \
      --out gt_only.present.chr1_5

## QC
plink --bfile gt_only.present.chr1_5 \
      --geno 0.05 \
      --mind 0.05 \
      --maf 0.05 \
      --make-bed \
      --out gt_only.present.chr1_5.qc

## LD Pruning
plink --bfile gt_only.present.chr1_5.qc \
      --indep-pairwise 50 5 0.2 \
      --out gt_only.present.chr1_5.qc

plink --bfile gt_only.present.chr1_5.qc \
      --extract gt_only.present.chr1_5.qc.prune.in \
      --make-bed \
      --out gt_only.present.chr1_5.qcpruned



In [ ]:
# PLINK IBD
plink --bfile gt_only.present.chr1_5.qcpruned \
      --genome full \
      --out plink_genome.present.chr1_5.qcpruned

In [ ]:
# GERMLINE
for c in 1 2 3 4 5
do
  echo "=== Make PED/MAP for chr${c} ==="
  plink --bfile gt_only.present.chr1_5.qcpruned \
        --chr ${c} \
        --recode 12 \
        --out gt_only.present.chr${c}.qcpruned.nomissSNP_ped
done

In [ ]:
for c in 1 2 3 4 5
do
  echo "=== GERMLINE chr${c} ==="
  germline \
    -input gt_only.present.chr${c}.qcpruned.nomissSNP_ped.ped gt_only.present.chr${c}.qcpruned.nomissSNP_ped.map \
    -output germline.chr${c}.qcpruned \
    -min_m 3 \
    -err_hom 0 \
    -err_het 2 \
    -bits 64
done

In [ ]:
cat germline.chr1.qcpruned.match germline.chr2.qcpruned.match germline.chr3.qcpruned.match germline.chr4.qcpruned.match germline.chr5.qcpruned.match \
  > germline.chr1_5.qcpruned.perchr_merged.match